# RPT API Reference

## Authentication

All API requests require authentication using your personal API token passed as a Bearer token in the `Authorization` header.

```bash
curl -X POST https://rpt.cloud.sap/api/predict \
  -H "Authorization: Bearer <your_token>" \
  -H "Content-Type: application/json" \
  -d '{"rows": [...]}'
```

---

## `POST /api/predict`

Submit data for prediction using in-context learning. The model learns from example rows (context rows) and predicts values for query rows that contain the `[PREDICT]` placeholder.

**Request body:**
```json
{
  "rows": [
    {"PRODUCT": "Laptop Pro",       "CATEGORY": "Electronics", "SALESGROUP": "Enterprise"},
    {"PRODUCT": "Standing Desk",    "CATEGORY": "Furniture",   "SALESGROUP": "Office"},
    {"PRODUCT": "Wireless Headset", "CATEGORY": "Electronics", "SALESGROUP": "[PREDICT]"}
  ],
  "index_column": "PRODUCT",
  "sampler_options": {"seed": 42},
  "prediction_config": {
    "target_columns": [{"name": "SALESGROUP", "task_type": "classification"}]
  },
  "explanations": {"top_column_scores": 4, "top_relevant_context_rows": 3}
}
```

**Parameters:**

| Parameter | Required | Description |
|---|---|---|
| `rows` | yes | Array of row objects. Context rows have complete data; query rows contain `[PREDICT]` |
| `index_column` | no | Column used as row identifier in the response |
| `sampler_options.seed` | no | Integer ≥ 0 — makes random downsampling reproducible |
| `prediction_config.target_columns[].task_type` | no | `"classification"` or `"regression"` — auto-detected if omitted |
| `explanations.top_column_scores` | no | Number of top feature importance scores to return (1–50, default 4) |
| `explanations.top_relevant_context_rows` | no | Number of most relevant context row indices to return (1–50, default 3) |

**Constraints:**

| | |
|---|---|
| Query rows | min 1, max 128 |
| Context rows | min 2, max 50,000 (auto-downsampled to 2,048 for model) |
| Recommended context | 500–2,000 rows |
| Target columns | max 10 |
| Total columns | max 100 |
| Characters per cell | max 1,000 |
| Characters per column name | max 100 |

**Response:**
```json
{
  "prediction": {
    "id": "6a2a7242-...",
    "status": {"code": 0, "message": "ok"},
    "metadata": {"num_columns": 3, "num_query_rows": 1, "num_rows": 3},
    "predictions": [
      {
        "PRODUCT": "Wireless Headset",
        "SALESGROUP": [{"prediction": "Enterprise", "confidence": 0.87, "confidence_interval": null}]
      }
    ],
    "explanations": {
      "top_column_scores": [{"CATEGORY": 0.74, "PRODUCT": 0.26}],
      "top_relevant_context_rows": [[0, 1]]
    }
  },
  "delay": 312.5,
  "samplingMetadata": {"downsampled": false, "sampledFrom": 2, "sampledTo": 2, "strategy": "random"}
}
```

---

## Context Downsampling

Requests with more than 2,048 context rows are randomly downsampled (Fisher-Yates) to 2,048 before being sent to the model. Query rows are never downsampled. Pass `sampler_options.seed` for reproducible results.

| | |
|---|---|
| Max context accepted | 50,000 rows |
| Max context sent to model | 2,048 rows |
| Strategy | random (Fisher-Yates) |

---

## Error codes

| Code | Meaning |
|---|---|
| `400` | Bad Request — invalid data format or validation error |
| `401` | Unauthorized — invalid or missing API token |
| `429` | Too Many Requests — rate limit exceeded; check `Retry-After` header |
| `503` | Service Unavailable — high load; retry after `Retry-After` header |
| `500` | Internal Server Error — contact support |
| `502` | Remote data source URL returned an error or could not be parsed as CSV |
| `504` | Remote data source fetch timed out (30s limit) |

---

## Rate limits

Requests are rate-limited. On `429`, the `Retry-After` header tells you when to retry.

---

## Available endpoints

Only `/api/predict` is accessible via API token. Chat-RPT, what-if simulations, and all conversational features are available exclusively through the web UI at [rpt.cloud.sap](https://rpt.cloud.sap).